In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import numpy as np
from tool import print_tensor_l2d

# --------------------------
# 1. 配置参数（可灵活调整）
# --------------------------
IMAGE_PATH = "./assets/Mafra_2013_720.png"  # 本地文件路径或URL
PATCH_SIZE = 240  # 每个patch的尺寸（720x720图片切成3x3，每个patch 240x240）
IN_CHANNELS = 3  # 图片通道数（RGB）
EMBED_DIM = 768  # 线性投影后的嵌入维度（ViT-Base默认768）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------
# 2. 图片读取与预处理
# --------------------------
def load_and_preprocess_image(image_path: str, target_size: int = 720) -> torch.Tensor:
    """
    读取图片并预处理为ViT输入格式
    :param image_path: 本地文件路径或URL
    :param target_size: 图片缩放后的尺寸（默认720x720，确保能被3x3分割）
    :return: 预处理后的tensor (1, C, H, W)
    """
    # 读取图片
    if image_path.startswith(("http://", "https://")): # 从URL下载图片
        response = requests.get(image_path, stream=True)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        # 从本地读取图片
        image = Image.open(image_path).convert("RGB")
    
    # 预处理流水线（缩放、归一化、转tensor）
    preprocess = transforms.Compose([
        transforms.Resize((target_size, target_size)),  # 缩放到720x720
        transforms.ToTensor(),  # 转为(C, H, W)，值范围[0,1]
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet均值
                             std=[0.229, 0.224, 0.225])   # ImageNet标准差
    ])
    
    return preprocess(image).unsqueeze(0).to(DEVICE)  # 添加batch维度

# --------------------------
# 3. ViT核心组件实现
# --------------------------
class PatchEmbedding(nn.Module):
    """ViT中的Patch Embedding层（分割+线性投影）"""
    def __init__(self, patch_size: int = PATCH_SIZE, in_channels: int = IN_CHANNELS, embed_dim: int = EMBED_DIM):
        """
        :param patch_size: 每个patch的尺寸
        :param in_channels: 输入通道数
        :param embed_dim: 嵌入维度
        :param log: 是否打印调试信息
        """
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        
        # HL: 线性投影层（等价于卷积核=patch_size、步长=patch_size的卷积）
        self.projection = nn.Conv2d(
            in_channels=in_channels,  # 输入通道数 3
            out_channels=embed_dim,  # 输出通道数 768
            kernel_size=patch_size,  # 卷积核大小 240x240
            stride=patch_size,  # 步长 240
            padding=0
        )
        # self.proj = nn.Linear(path_size*patch_size*in_channels, embed_dim)
        
        # Class Token（ViT的特殊token）
        self.class_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        
    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, int]:
        """
        前向传播：分割patch + 线性投影 + 添加class token
        :param x: 输入图片 (batch_size, C, H, W)
        :return: (embed + class_token, num_patches)
        """
        batch_size = x.shape[0]
        
        # 1. 分割并投影patch: (B, C, H, W) -> (B, embed_dim, num_patches_h, num_patches_w)
        x = self.projection(x)  # 输出形状: (B, 768, 3, 3)
        num_patches_h, num_patches_w = x.shape[2], x.shape[3]
        num_patches = num_patches_h * num_patches_w
        
        # 2. 展平为序列: (B, embed_dim, H, W) -> (B, num_patches, embed_dim)
        x = x.permute(0, 2, 3, 1).reshape(batch_size, num_patches, self.embed_dim)
        
        # 3. 添加class token: (B, 1, embed_dim) -> 拼接后 (B, num_patches+1, embed_dim)
        class_tokens = self.class_token.expand(batch_size, -1, -1)  # 扩展到batch维度
        x = torch.cat([class_tokens, x], dim=1)
        
        return x, num_patches

class PositionalEncoding(nn.Module):
    """ViT中的位置编码（可学习参数）"""
    def __init__(self, embed_dim: int = EMBED_DIM, num_patches: int = 9):
        super().__init__()
        # 位置编码：num_patches + 1（包含class token的位置）
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, embed_dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        前向传播：添加位置编码
        :param x: (B, num_patches+1, embed_dim)
        :return: (B, num_patches+1, embed_dim)
        """
        return x + self.pos_embed

# --------------------------
# 4. 可视化工具（可选）
# --------------------------
def visualize_patches(image_tensor: torch.Tensor, patch_size: int = PATCH_SIZE):
    """
    可视化分割后的9个patch
    :param image_tensor: 预处理后的图片 (1, C, H, W)
    :param patch_size: patch尺寸
    """
    # 反归一化，便于可视化
    inv_normalize = transforms.Normalize(
        mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
        std=[1/0.229, 1/0.224, 1/0.225]
    )
    
    image = inv_normalize(image_tensor.squeeze(0)).permute(1, 2, 0).cpu().numpy()
    image = np.clip(image, 0, 1)  # 确保像素值在[0,1]范围
    
    # 创建3x3子图
    fig, axes = plt.subplots(3, 3, figsize=(6, 6))
    axes = axes.flatten()
    
    # 分割并显示每个patch
    for i in range(3):
        for j in range(3):
            # 计算patch的坐标
            h_start = i * patch_size
            h_end = h_start + patch_size
            w_start = j * patch_size
            w_end = w_start + patch_size
            
            patch = image[h_start:h_end, w_start:w_end, :]
            axes[i*3 + j].imshow(patch)
            axes[i*3 + j].set_title(f"Patch {i*3 + j + 1}")
            axes[i*3 + j].axis("off")
    
    plt.tight_layout()
    plt.show()

# --------------------------
# 5. 主流程执行
# --------------------------
if __name__ == "__main__":
    # 步骤1：读取并预处理图片
    print(f"正在读取图片: {IMAGE_PATH}")
    image = load_and_preprocess_image(IMAGE_PATH)
    print(f"图片预处理后形状: {image.shape} (batch, channels, height, width)")
    
    # 步骤2：初始化ViT组件
    patch_embed = PatchEmbedding().to(DEVICE)
    pos_encoding = PositionalEncoding().to(DEVICE)
    
    # 步骤3：执行patch分割、线性投影、位置编码
    print("执行patch分割与线性投影...")
    patch_embeds, num_patches = patch_embed(image)
    print_tensor_l2d(patch_embed.projection.weight)
    print(f"Patch嵌入后形状: {patch_embeds.shape} (batch, num_tokens, embed_dim)")
    print(f"分割的patch数量: {num_patches}")
    
    print("添加位置编码...")
    final_embeds = pos_encoding(patch_embeds)
    print(f"最终嵌入形状: {final_embeds.shape} (batch, num_tokens, embed_dim)")
    
    # 步骤4：可视化分割后的patch（可选，注释掉可跳过）
    # print("可视化分割后的patch...")
    # visualize_patches(image)
    
    # 输出关键信息
    print(f"=== 最终结果 ===")
    print(f"输入图片尺寸: 720x720")
    print(f"Patch尺寸: {PATCH_SIZE}x{PATCH_SIZE}")
    print(f"Patch数量: {num_patches} (3x3)")
    print(f"嵌入维度: {EMBED_DIM}")
    print(f"最终输出形状: (batch_size, {num_patches+1}, {EMBED_DIM})")  # +1 for class token

正在读取图片: ./assets/Mafra_2013_720.png
图片预处理后形状: torch.Size([1, 3, 720, 720]) (batch, channels, height, width)
执行patch分割与线性投影...
【Tensor基础信息】
形状: torch.Size([768, 3, 240, 240]), 数据类型: torch.float32, 设备: cpu
------------------------------------------------------------
【前置维度（仅显示第一个）】维度0: 0/768 | 维度1: 0/3
【最后二维（最多10×10）】形状: torch.Size([240, 240])
------------------------------------------------------------
数据（前10行/240 | 前10列/240）:
行 0: 0.001901 0.001271 0.002270 -0.000718 -0.001602 -0.002023 -0.000976 0.001204 0.000810 0.001830 ...
行 1: 0.002109 -0.000641 -0.000120 0.001027 -0.001954 0.001013 -0.000099 0.000941 -0.001686 -0.001478 ...
行 2: 0.001605 0.002141 -0.000255 0.001504 -0.000986 -0.001060 -0.000599 0.001881 -0.000099 -0.001658 ...
行 3: -0.000551 0.002249 -0.001536 -0.000647 -0.000920 0.000582 -0.001751 -0.001791 -0.001565 -0.000663 ...
行 4: 0.000113 0.001404 -0.001794 -0.000911 0.000117 -0.000016 0.000350 -0.001601 0.000903 -0.000746 ...
行 5: -0.002023 -0.001976 -0.000720 -0.001468 -0